# Explicit chemical-template transfer — 2026-10-06

DockingMT #42 qualification with committed MolSysMT `5bd893c85fe8d211663b2b1f865f5f1d2c382a90`, ArgDigest 0.15.0 and the existing Python 3.14 CI viewer pin. Public operations preserve template history; the snapshot records public topology/structure domains and ChemicalStatesDict with fixed nm/ps units. H5MSM recovery is checked under pm/fs. Constructed 5X72 controls and original 181L BNZ do not establish docking readiness. Earlier notebook/JSON evidence remains unchanged. See [the contract](chemical_template_consumption.md).

In [1]:
import json
from pathlib import Path
import runpy
import sys

assert sys.version_info[:2] == (3, 14), "Select the molsyssuite@uibcdf_3.14 kernel"
assert "molsyssuite@uibcdf_3.14" in sys.executable

import molsysmt as msm
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'devtools/qualify_chemical_templates.py').is_file())
ns = runpy.run_path(str(root / 'devtools/qualify_chemical_templates.py'))
print(json.dumps({'python': sys.version.split()[0], 'interpreter': sys.executable,
                  'provider_import': msm.__file__,
                  'reference_source': ns['PROVIDER_REVISION']}, indent=2))

{
  "python": "3.14.7",
  "interpreter": "/home/liliana/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
  "provider_import": "/home/liliana/repos@uibcdf/molsysmt/molsysmt/__init__.py",
  "reference_source": "5bd893c85fe8d211663b2b1f865f5f1d2c382a90"
}


## Execute the bounded composition

The command verifies unchanged source/template data, atom identities, all poses,
box, time and frame/state associations. It explicitly chooses the second 5X72
frame before preparation. Reports retain original producer versions and declared
units; template hashes identify actual native snapshots separately from source
file hashes. This writes only this qualification's evidence file.

In [2]:
record = ns['qualify']()
path = ns['OUTPUT']
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(record, indent=2) + '\n')
print('Retained evidence:', path.relative_to(root))
print('Executed adapter versions:', record['versions'])
rows = []
for case in record['cases']:
    report = case['application_report']
    preparation = case['prepared_ligand']
    rows.append({'case': case['case'], 'status': report['status'],
                 'mapped atoms': report['source']['n_atoms'],
                 'assigned fields': len(report['assigned_fields']),
                 'preserved fields': len(report['preserved_fields']),
                 'H policy': report['coverage']['hydrogen_policy'],
                 'retained atoms': preparation['n_atoms']})
pd.DataFrame(rows)

Retained evidence: devguide/validation/data/chemical_templates/cases_2026-10-06.json
Executed adapter versions: {'molsysmt': '0.22.4+215.g5bd893c85', 'argdigest': '0.15.0+1.g5c6711e', 'pyunitwizard': '0.28.1+3.g2ab37a5', 'rdkit': '2026.3.1', 'numpy': '2.4.6', 'pandas': '2.3.3', 'vina': '1.2.7'}


,case,status,mapped atoms,assigned fields,preserved fields,H policy,retained atoms
0,5X72 P59 permuted absent-field control,applied,39,445,0,explicit_atoms,25
1,5X72 P69 permuted absent-field control,applied,39,445,0,explicit_atoms,25
2,181L original BNZ heavy-atom source,applied,6,54,12,stored_counts,6


## Rejections remain observable

Unchanged native 5X72 SDF and adapter templates require aromatic representation
normalization that this API does not provide. Explicit charge/stereo conflicts
and a missing stored edge fail before any mutation. The command tests application
failure as well as assessment. The regression suite additionally covers incomplete
H maps, matching explicit assignments and preservation of an unselected state.

In [3]:
rows = []
for case in record['cases']:
    if 'original_sdf_assessment' in case:
        report = case['original_sdf_assessment']
        rows.append({'case': case['case'].split(' permuted')[0] + ' original SDF',
                     'status': report['status'],
                     'reasons': sorted({issue['reason_code'] for issue in report['issues']})})
for case in record['negative_controls']:
    report = case['assessment']
    rows.append({'case': case['case'], 'status': report['status'],
                 'reasons': sorted({issue['reason_code'] for issue in report['issues']})})
pd.DataFrame(rows)

,case,status,reasons
0,5X72 P59 original SDF,unassessed,[aromatic_representation_requires_normalization]
1,5X72 P69 original SDF,unassessed,[aromatic_representation_requires_normalization]
2,charge,conflict,[explicit_assignment_conflict]
3,enantiomer,conflict,[explicit_assignment_conflict]
4,missing-edge,conflict,"[isolated_connected_component_required, stored..."


## Retain native history and the preparation limit

MolSysMT retains template reports in ChemicalStates and H5MSM 0.5. Snapshot schema `dockingmt.chemical_template_snapshot@2` retains that history without loss authorization to MolSysDict 0.1. The recovered topology, chemistry, history and associations match exactly; the two empty component text columns acquire nullable string dtypes; structure values use an absolute `1e-12 nm` tolerance for unit conversion. Nonempty MolecularMechanics is outside H5MSM 0.5 support. DockingMT retains the report alongside its preparation record; the docking-result summary still does not certify template chemistry, named charges or AutoDock types.

In [4]:
rows = []
for case in record['cases']:
    meta = case['prepared_ligand']['metadata']
    coverage = meta['source_chemistry']['chemical_readiness']
    assert 'docking_readiness' in coverage['unassessed_checks']
    assert meta['charge_source'] == 'zero_placeholder'
    rows.append({'case': case['case'],
                 'formal-charge coverage': coverage['fields']['formal_charge']['status'],
                 'partial-charge source': meta['charge_source'],
                 'AutoDock typing': meta['atom_type_source'],
                 'template report': case['application_report']['schema']})
pd.DataFrame(rows)

,case,formal-charge coverage,partial-charge source,AutoDock typing,template report
0,5X72 P59 permuted absent-field control,present,zero_placeholder,element_aromaticity_heuristic,molsysmt.chemical_template@1
1,5X72 P69 permuted absent-field control,present,zero_placeholder,element_aromaticity_heuristic,molsysmt.chemical_template@1
2,181L original BNZ heavy-atom source,present,zero_placeholder,element_aromaticity_heuristic,molsysmt.chemical_template@1


In [5]:
recovery = record['history_recovery']
assert recovery['original_preparation_history'] == recovery['recovered_preparation_history']
assert recovery['topology_chemistry_history_and_associations_preserved']
assert recovery['structures_preserved_within_tolerance']
print(recovery['format'], recovery['snapshot_schema'], recovery['structure_units'])
print('Retained preparation records:', len(recovery['recovered_preparation_history']))

H5MSM 0.5 dockingmt.chemical_template_snapshot@2 {'coordinates': 'nm', 'box': 'nm', 'time': 'ps'}
Retained preparation records: 2
